# D0 — Setup, load data & the best pipeline

-Imports libs

-Ensures artifacts/ exists

-Tries to reuse in-memory objects (df_model, price_pipeline_best); if missing, loads them from disk.

In [13]:
# D0 — Setup, load data & the best pipeline

from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from joblib import load, dump
from datetime import datetime

ARTIFACTS = Path("artifacts")
ARTIFACTS.mkdir(exist_ok=True, parents=True)

# 0) Load the SAME engineered file you used in the modeling notebook
DATA_PATH = Path("data/master_cars_with_features.csv")
df_model = pd.read_csv(DATA_PATH, encoding="utf-8-sig")

# 1) Sanity: we only keep the columns needed here
need_cols = [
    "brand", "series_auto",
    "year_num", "car_age",
    "mileage_km_num", "mileage_per_year",
    "price_thb_num"  # ground truth for optional hedonic fit; OK if missing later
]
missing = [c for c in need_cols if c not in df_model.columns]
if missing:
    raise ValueError(f"Your data file is missing columns: {missing}\n"
                     f"Make sure you're loading data/master_cars_with_features.csv")

# 2) Load the champion pipeline (produced in T3.6)
#    Falls back to the older RF pipeline name if needed.
PIPE_MAIN = ARTIFACTS / "price_pipeline_best.joblib"
PIPE_FALLBACK = Path("models/price_model_rf_pipeline.joblib")

if PIPE_MAIN.exists():
    price_pipeline_best = load(PIPE_MAIN)
elif PIPE_FALLBACK.exists():
    price_pipeline_best = load(PIPE_FALLBACK)
else:
    raise FileNotFoundError(
        "Could not find artifacts/price_pipeline_best.joblib "
        "or models/price_model_rf_pipeline.joblib. Re-run T3.6."
    )

print("✅ df_model shape:", df_model.shape)
print("✅ Pipeline:", type(price_pipeline_best.named_steps["model"]).__name__)


✅ df_model shape: (6805, 49)
✅ Pipeline: XGBRegressor


# D1 — Recreate Tier-1 features (exactly like the training notebook)

These helpers mirror the transformations you used during training: rare brand/series bucketing, log-mileage, age buckets, and interactions. We compute maps from your data so they’re stable.

In [14]:
# D1 — Tier-1 feature builders (vectorized)

def build_tier_maps(df: pd.DataFrame,
                    brand_thresh: int = 20,
                    series_thresh: int = 10):
    b_counts = df["brand"].value_counts()
    s_counts = df["series_auto"].value_counts()

    brand_map = {b: (b if cnt >= brand_thresh else "OTHER") for b, cnt in b_counts.items()}
    series_map = {s: (s if cnt >= series_thresh else "OTHER_SERIES") for s, cnt in s_counts.items()}

    return brand_map, series_map

BRAND_TIER1_MAP, SERIES_TIER1_MAP = build_tier_maps(df_model)

def age_bucket_from_age(age: float) -> str:
    if age <= 3:   return "0_3_new"
    if age <= 7:   return "4_7_mid"
    if age <= 12:  return "8_12_old"
    return "12_plus"

def add_tier1_features(df_in: pd.DataFrame) -> pd.DataFrame:
    """
    Returns a new DataFrame with Tier-1 features the model expects:
    - brand_tier1, series_tier1, age_bucket
    - log_mileage, age_x_mileage, age_x_log_mileage, mileage_intensity
    """
    df = df_in.copy()

    # brand/series tiering (match training)
    df["brand_tier1"]  = df["brand"].map(BRAND_TIER1_MAP).fillna("OTHER")
    df["series_tier1"] = df["series_auto"].map(SERIES_TIER1_MAP).fillna("OTHER_SERIES")

    # derived
    df["log_mileage"]        = np.log(df["mileage_km_num"].astype(float) + 1.0)
    df["age_bucket"]         = df["car_age"].astype(float).apply(age_bucket_from_age)
    df["age_x_mileage"]      = df["car_age"].astype(float) * df["mileage_km_num"].astype(float)
    df["age_x_log_mileage"]  = df["car_age"].astype(float) * df["log_mileage"].astype(float)
    df["mileage_intensity"]  = df["mileage_km_num"].astype(float) / (df["car_age"].astype(float) + 1.0)

    # trim_final wasn't required for depreciation; include as empty if missing
    if "trim_final" not in df.columns:
        df["trim_final"] = ""

    return df


# D2 — Robust, API-ready predictors

predict_prices_bulk() takes a DataFrame with raw columns (brand, series_auto, year_num, car_age, mileage_km_num, mileage_per_year) and returns model predictions.

predict_price_single() is a convenience wrapper for API use.

In [15]:
# D2 — Predictors (vectorized and single-row), API ready

def predict_prices_bulk(df_rows: pd.DataFrame) -> np.ndarray:
    """Vectorized predictions with Tier-1 feature rebuild to mirror training."""
    # Ensure all raw columns exist
    raw_need = ["brand","series_auto","year_num","car_age","mileage_km_num","mileage_per_year"]
    miss = [c for c in raw_need if c not in df_rows.columns]
    if miss:
        raise ValueError(f"predict_prices_bulk missing columns: {miss}")

    X = add_tier1_features(df_rows)
    y_hat = price_pipeline_best.predict(X)
    return y_hat

def predict_price_single(brand: str, series_auto: str, year_num: int,
                         car_age: float, mileage_km_num: float,
                         mileage_per_year: float = 0.0) -> float:
    row = pd.DataFrame([{
        "brand": brand, "series_auto": series_auto,
        "year_num": year_num, "car_age": car_age,
        "mileage_km_num": mileage_km_num, "mileage_per_year": mileage_per_year
    }])
    return float(predict_prices_bulk(row)[0])


# D3 — General depreciation curves (per brand/series at a reference mileage)

For each age in the group, we predict price at a fixed mileage reference (median or a value you pass).

We compute YoY % drop from the model-based curve.

Optionally, we fit a tiny hedonic regression (log(price) ~ age + log(mileage)) and extract % per extra year for easy interpretation. (Auto-fallback to numpy regression if statsmodels is not present.)

In [16]:
# D3 — Build depreciation curves with optional hedonic slope

def _hedonic_per_year_pct(grp: pd.DataFrame) -> float | None:
    """
    Estimate interpretable % per year using log-linear hedonic model on the group.
    Returns per-year % (negative is drop), or None if cannot fit.
    """
    # Need price and core features available
    need = ["price_thb_num","car_age","mileage_km_num"]
    if any(c not in grp.columns for c in need):
        return None
    g = grp.dropna(subset=need).copy()
    if len(g) < 50:  # guard: too small groups are noisy
        return None

    g["log_price"]    = np.log(g["price_thb_num"].astype(float) + 1.0)
    g["log_mileage"]  = np.log(g["mileage_km_num"].astype(float) + 1.0)

    # Try statsmodels if available
    try:
        import statsmodels.api as sm
        X = g[["car_age","log_mileage"]].astype(float)
        X = sm.add_constant(X)
        y = g["log_price"].astype(float)
        model = sm.OLS(y, X).fit()
        beta_age = float(model.params["car_age"])
    except Exception:
        # Fallback: simple numpy regression on log_price ~ const + age + log_mileage
        try:
            X = np.c_[np.ones(len(g)), g["car_age"].values, g["log_mileage"].values]
            beta = np.linalg.lstsq(X, g["log_price"].values, rcond=None)[0]
            beta_age = float(beta[1])
        except Exception:
            return None

    # Interpretable: exp(beta_age) - 1 ≈ % change per +1 year (at ref mileage)
    return (np.exp(beta_age) - 1.0) * 100.0

def build_general_curve(df_all: pd.DataFrame,
                        brand: str,
                        series_auto: str,
                        mileage_ref: str | float = "median") -> dict | None:
    """
    Returns a dict with arrays for ages and predicted prices at a fixed mileage reference:
      {
        "brand": str,
        "series_auto": str,
        "age": [0,1,2,...],
        "price_at_ref_mileage": [...],
        "yoy_pct": [...],   # % drop from t->t+1, length len(age)-1
        "hedonic_pct_per_year": float | None
      }
    """
    grp = df_all.loc[(df_all["brand"] == brand) & (df_all["series_auto"] == series_auto)].copy()
    if grp.empty:
        return None

    # Determine mileage reference
    if mileage_ref == "median":
        ref_m = float(grp["mileage_km_num"].median())
    else:
        ref_m = float(mileage_ref)

    # Use the observed age span (clean)
    ages = sorted(int(a) for a in grp["car_age"].dropna().unique())
    if len(ages) < 2:
        return None

    # Build rows for predictions at fixed mileage
    rows = []
    for a in ages:
        # pick a plausible year_num for that age (most common for that age)
        y_mode = grp.loc[grp["car_age"] == a, "year_num"].mode()
        year_num = int(y_mode.iloc[0]) if not y_mode.empty else int(grp["year_num"].median())
        rows.append({
            "brand": brand, "series_auto": series_auto,
            "year_num": year_num, "car_age": float(a),
            "mileage_km_num": ref_m, "mileage_per_year": 0.0
        })
    Xgrid = pd.DataFrame(rows)
    p_hat = predict_prices_bulk(Xgrid)

    # YoY % drop from curve
    yoy = []
    for i in range(len(p_hat) - 1):
        if p_hat[i] > 0:
            yoy.append( (p_hat[i+1] - p_hat[i]) / p_hat[i] * 100.0 )
        else:
            yoy.append(np.nan)

    hedonic_pct = _hedonic_per_year_pct(grp)

    return {
        "brand": brand,
        "series_auto": series_auto,
        "age": ages,
        "price_at_ref_mileage": [float(x) for x in p_hat],
        "yoy_pct": yoy,
        "hedonic_pct_per_year": None if hedonic_pct is None else float(hedonic_pct),
    }


# D4 — Generate & export curves (global and per brand/series)

One global “general” curve (median mileage overall)

Per-brand+series curves with the group median mileage (default).

Write CSV exports and a single pickle for serving.

In [17]:
# D4 — Build & save curves

# 1) Global "general" curve by using the full dataset's median mileage
global_curves = []
for (b, s), grp in df_model.groupby(["brand","series_auto"]):
    curve = build_general_curve(df_model, b, s, mileage_ref="median")
    if curve is not None:
        global_curves.append(curve)

print(f"✅ Built {len(global_curves)} brand+series curves")

# 2) Save CSV: flattened per-point table for easy plotting later
def curves_to_table(curves: list[dict]) -> pd.DataFrame:
    records = []
    for c in curves:
        for i, a in enumerate(c["age"]):
            rec = {
                "brand": c["brand"],
                "series_auto": c["series_auto"],
                "age": a,
                "price_at_ref_mileage": c["price_at_ref_mileage"][i],
                "hedonic_pct_per_year": c["hedonic_pct_per_year"],
            }
            # yoy_pct is defined between points; attach for i>0
            if i < len(c["yoy_pct"]):
                rec["yoy_pct_next"] = c["yoy_pct"][i]
            else:
                rec["yoy_pct_next"] = np.nan
            records.append(rec)
    return pd.DataFrame(records)

tbl = curves_to_table(global_curves).sort_values(["brand","series_auto","age"])
csv_path = ARTIFACTS / "depreciation_curves_v2_model.csv"
tbl.to_csv(csv_path, index=False, encoding="utf-8")
print(f"💾 Saved curves table → {csv_path} ({len(tbl):,} rows)")

# 3) Also save a compact pickle with the raw curves list (API can load this fast)
pkl_path = ARTIFACTS / "depreciation_curves_v2_model.joblib"
dump(global_curves, pkl_path)
print(f"💾 Saved curves object  → {pkl_path}")


✅ Built 206 brand+series curves
💾 Saved curves table → artifacts\depreciation_curves_v2_model.csv (1,534 rows)
💾 Saved curves object  → artifacts\depreciation_curves_v2_model.joblib


# D5 — API-ready helpers (import these in FastAPI/Flask)

These are pure functions. You can import them from this notebook (or copy into a .py) and call directly inside your web service.

In [18]:
# D5 — API-ready helpers

def api_curve(brand: str, series_auto: str,
              mileage_ref: str | float = "median") -> dict:
    """
    Returns a single curve dict (see build_general_curve) for one brand+series.
    Raises ValueError if brand/series not present.
    """
    if df_model.loc[(df_model["brand"] == brand) & (df_model["series_auto"] == series_auto)].empty:
        raise ValueError(f"No data for brand='{brand}', series_auto='{series_auto}'")

    c = build_general_curve(df_model, brand, series_auto, mileage_ref=mileage_ref)
    if c is None:
        raise ValueError("Unable to build a curve (not enough data).")
    return c

def api_yearly_drop(brand: str, series_auto: str,
                    year_num: int, car_age: float,
                    mileage_km_num: float, mileage_per_year: float = 0.0) -> dict:
    """
    Returns price_now, price_next_year, and deltas based on the model.
    """
    now = predict_price_single(brand, series_auto, year_num, car_age, mileage_km_num, mileage_per_year)
    nxt = predict_price_single(brand, series_auto, year_num, car_age + 1.0, mileage_km_num, mileage_per_year)
    return {
        "price_now": now,
        "price_next_year": nxt,
        "abs_change": nxt - now,
        "pct_change": (nxt - now) / now * 100.0 if now > 0 else np.nan
    }


# (Optional) D6 — Tiny FastAPI skeleton that uses the helpers

This is just to show it’s plug-and-play. You can paste this into api.py and run uvicorn api:app --reload.

In [19]:
# D6 — Optional: FastAPI skeleton (run in a separate file)
"""
from fastapi import FastAPI, HTTPException

app = FastAPI()

@app.get("/curve")
def curve(brand: str, series_auto: str, mileage_ref: float | str = "median"):
    try:
        return api_curve(brand, series_auto, mileage_ref)
    except Exception as e:
        raise HTTPException(status_code=400, detail=str(e))

@app.get("/yearly_drop")
def yearly_drop(brand: str, series_auto: str, year_num: int,
                car_age: float, mileage_km_num: float, mileage_per_year: float = 0.0):
    try:
        return api_yearly_drop(brand, series_auto, year_num, car_age, mileage_km_num, mileage_per_year)
    except Exception as e:
        raise HTTPException(status_code=400, detail=str(e))
"""


'\nfrom fastapi import FastAPI, HTTPException\n\napp = FastAPI()\n\n@app.get("/curve")\ndef curve(brand: str, series_auto: str, mileage_ref: float | str = "median"):\n    try:\n        return api_curve(brand, series_auto, mileage_ref)\n    except Exception as e:\n        raise HTTPException(status_code=400, detail=str(e))\n\n@app.get("/yearly_drop")\ndef yearly_drop(brand: str, series_auto: str, year_num: int,\n                car_age: float, mileage_km_num: float, mileage_per_year: float = 0.0):\n    try:\n        return api_yearly_drop(brand, series_auto, year_num, car_age, mileage_km_num, mileage_per_year)\n    except Exception as e:\n        raise HTTPException(status_code=400, detail=str(e))\n'